In [1]:
import pandas as pd
import pathlib
import sys
import matplotlib.pyplot as plt
import numpy as np
import gensim
from collections import defaultdict
from gensim import corpora
import re
from gensim.models import LdaModel
import spacy
from spacy.tokenizer import Tokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation
from pathlib import Path
import sys
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
nlp = spacy.load("fr_core_news_lg")
nlp_trf = spacy.load("fr_dep_news_trf")

In [42]:
dossier_path = Path("corpus_goncourt")

donnees = []

# On boucle sur tous les fichiers .txt du dossier
for fichier in dossier_path.glob("*.txt"):
    with open(fichier, "r", encoding="utf-8") as f:
        contenu = f.read()
        
        # On détermine le label selon le nom du fichier
    donnees.append({
            "nom_fichier": fichier.name,
            "texte_brut": contenu,
        })
      
         

# Création du tableau de bord (DataFrame)
df = pd.DataFrame(donnees)
print(f"{len(df)} textes chargés avec succès.")

10 textes chargés avec succès.


In [43]:
df

,nom_fichier,texte_brut
0,1869_madame_gervaisais_travail.txt,"— Quarante scudi ?\n— Oui, signora.\n— Cela fa..."
1,1884_cherie_travail.txt,"petites amies à peu près de son âge, des place..."
2,1861_sœur_philomène_travail.txt,"La salle est haute et vaste. Elle est longue, ..."
3,1879_frères_zemganno_travail.txt,"En pleine campagne, au pied d’un poteau d’octr..."
4,1877_la_fille_elisa_travail.txt,"La femme, la prostituée condamnée à mort, étai..."
5,1865_germinie_lacerteux_travail.txt,"— Sauvée ! vous voilà donc sauvée, mademoisell..."
6,1867_manette_salomon_travail.txt,On était au commencement de novembre. La derni...
7,1860_Charles Demailly_travail.txt,– Un article ?… Tu me demandes s’ y a un artic...
8,1882_la_faustin_travail.txt,"faisait nuit sous un ciel étoilé, au-dessus d'..."
9,1864_renée_mauperin_travail.txt,"— Vous n’aimez pas le monde, mademoiselle ?\n—..."


In [44]:
def segmenter_texte(text, taille=400):
    mots = text.split()
    segments = []
    for i in range(0, len(mots), taille):
        segment = " ".join(mots[i:i+taille])
        if len(segment.split()) >= 100:  # évite les segments trop courts à la fin
            segments.append(segment)
    return segments

In [46]:
lignes = []

for _, row in df.iterrows():
    titre = row["nom_fichier"]
    texte = row["texte_brut"]
    segments = segmenter_texte(texte, taille=400)
    
    for j, seg in enumerate(segments):
        lignes.append({
            "titre": titre,
            "segment_id": j,
            "texte_segment": seg
        })

df_segments = pd.DataFrame(lignes)

In [ ]:
df_segments.head()

,titre,segment_id,texte_segment
0,1869_madame_gervaisais_travail.txt,0,"— Quarante scudi ? — Oui, signora. — Cela fait..."
1,1869_madame_gervaisais_travail.txt,1,d’étrangers à Rome… — Dites-moi : la maison es...
2,1869_madame_gervaisais_travail.txt,2,"Je dois vous prévenir pour les scarpe, les sou..."
3,1869_madame_gervaisais_travail.txt,3,place des touristes consciencieux lisaient le ...
4,1869_madame_gervaisais_travail.txt,4,"Des paroles professorales, de grossières ignor..."
...,...,...,...
1708,1864_renée_mauperin_travail.txt,163,un vilain papa qui me trouve toujours mauvaise...
1709,1864_renée_mauperin_travail.txt,164,!… Oh ! je voudrais que ce fût demain… Je n’en...
1710,1864_renée_mauperin_travail.txt,165,disaient bientôt plus rien ; ils restaient mue...
1711,1864_renée_mauperin_travail.txt,166,"le cadre était incliné, semblait se pencher su..."


In [55]:
def nettoyer_texte(texte):
    doc = nlp(texte)
    tokens = []
    
    for token in doc:
        if (
            not token.is_stop
            and not token.is_punct
            and not token.like_num
            and not token.is_space
            and token.pos_ in {"NOUN", "ADJ", "VERB"}
            and len(token.lemma_) > 2
        ):
            tokens.append(token.lemma_.lower())
    
    return " ".join(tokens)

In [ ]:
df_segments["texte_nettoye"] = df_segments["texte_segment"].apply(nettoyer_texte)

In [50]:
df_segments[["texte_segment", "texte_nettoye"]].head()

,texte_segment,texte_nettoye
0,"— Quarante scudi ? — Oui, signora. — Cela fait...",scudi signorer monnaie france cent franc cent ...
1,d’étrangers à Rome… — Dites-moi : la maison es...,étranger rome dite maison tranquille bruit heu...
2,"Je dois vous prévenir pour les scarpe, les sou...",devoir prévenir scarpe soulier femme chambre f...
3,place des touristes consciencieux lisaient le ...,place touriste consciencieux lire guide assiet...
4,"Des paroles professorales, de grossières ignor...",parole professoral grossier ignorance critique...


In [51]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_df=0.8,
    min_df=5
)

X = vectorizer.fit_transform(df_segments["texte_nettoye"])

In [52]:
X.shape

(1713, 6068)

In [53]:
from sklearn.decomposition import NMF

nmf = NMF(n_components=8, random_state=42)
W = nmf.fit_transform(X)
H = nmf.components_

In [54]:
feature_names = vectorizer.get_feature_names_out()

def afficher_topics(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
        print(f"Topic {topic_idx+1} : {' | '.join(top_words)}")

afficher_topics(nmf, feature_names, 10)

Topic 1 : germinie | mademoiselle | jupillon | varandeuil | voir | mlle | vouloir | faire | aller | bon
Topic 2 : mauperin | monsieur | renée | denoisel | bourjot | henri | noémi | faire | voir | savoir
Topic 3 : coriolis | anatole | manette | atelier | faire | garnotelle | voir | tableau | crescent | artiste
Topic 4 : petit | blanc | grand | noir | ombre | ciel | eau | bleu | arbre | lumière
Topic 5 : femme | fille | homme | amour | jeune | vie | petit | aimer | monde | grand
Topic 6 : gianni | frère | nello | cirque | exercice | saut | pied | tour | clown | directeur
Topic 7 : charles | marthe | nachette | couturat | journal | homme | demailly | savoir | faire | cher
Topic 8 : sœur | barnier | malade | lit | philomène | salle | interne | hôpital | faire | malivoire
